In [1]:
!pip install gradio

In [3]:
import gradio as gr
from datetime import date
import re

# =============================================
#         INDUSTRY TEMPLATES
# =============================================

industry_data = {
    "Tech / Software": {
        "market_size": "$5.2 Trillion global IT market (2024)",
        "trends": "AI, Cloud Computing, Cybersecurity, SaaS, Automation",
        "competitors": ["Google", "Microsoft", "Salesforce", "startups in AI/SaaS space"],
        "revenue_models": ["Subscription (SaaS)", "Freemium", "API Licensing", "Enterprise Contracts"],
        "risks": ["Rapid tech obsolescence", "Data privacy regulations", "High competition"],
        "kpis": ["Monthly Active Users (MAU)", "Churn Rate", "Customer Acquisition Cost (CAC)", "MRR/ARR"],
    },
    "Healthcare / MedTech": {
        "market_size": "$665 Billion digital health market by 2028",
        "trends": "Telemedicine, AI Diagnostics, Wearables, Electronic Health Records",
        "competitors": ["Epic Systems", "Philips Healthcare", "Teladoc", "local health startups"],
        "revenue_models": ["B2B Hospital Contracts", "Insurance Tie-ups", "Per-consultation Fee", "Subscription"],
        "risks": ["Regulatory approvals (FDA/CE)", "Patient data privacy (HIPAA)", "Slow adoption"],
        "kpis": ["Patient Outcomes", "Cost per Diagnosis", "Clinician Adoption Rate", "Revenue per Bed"],
    },
    "Finance / FinTech": {
        "market_size": "$340 Billion FinTech market by 2026",
        "trends": "Digital Payments, Blockchain, Robo-advisors, BNPL, Open Banking",
        "competitors": ["PayPal", "Stripe", "Razorpay", "regional neobanks"],
        "revenue_models": ["Transaction Fees", "Interest Margin", "Premium Subscriptions", "Data Licensing"],
        "risks": ["Financial regulations (RBI/SEC)", "Fraud & cybersecurity", "Customer trust"],
        "kpis": ["Transaction Volume", "Net Promoter Score (NPS)", "Default Rate", "AUM"],
    },
    "Retail / E-Commerce": {
        "market_size": "$8.1 Trillion global e-commerce market by 2026",
        "trends": "D2C Brands, Quick Commerce, AR Shopping, Hyper-personalization",
        "competitors": ["Amazon", "Flipkart", "Shopify stores", "local D2C brands"],
        "revenue_models": ["Product Margins", "Marketplace Commission", "Subscription Box", "Advertising"],
        "risks": ["Logistics & supply chain", "High return rates", "Price wars"],
        "kpis": ["Gross Merchandise Value (GMV)", "Cart Abandonment Rate", "Repeat Purchase Rate", "LTV"],
    },
    "Education / EdTech": {
        "market_size": "$400 Billion EdTech market by 2025",
        "trends": "Microlearning, Gamification, AI Tutors, Skill-based Learning, VR Classrooms",
        "competitors": ["Coursera", "Byju's", "Unacademy", "local coaching platforms"],
        "revenue_models": ["Course Fees", "Certification Programs", "B2B Corporate Training", "Freemium"],
        "risks": ["Student retention", "Content quality", "Regulatory compliance"],
        "kpis": ["Course Completion Rate", "Student NPS", "Revenue per Learner", "DAU"],
    },
    "Sustainability / GreenTech": {
        "market_size": "$17 Trillion climate tech investment opportunity by 2030",
        "trends": "Carbon Credits, EV Infrastructure, Renewable Energy, Circular Economy",
        "competitors": ["Tesla Energy", "Siemens Gamesa", "local solar startups"],
        "revenue_models": ["Carbon Credit Trading", "Energy-as-a-Service", "Government Grants", "B2B Contracts"],
        "risks": ["Policy dependency", "High capex", "Slow ROI"],
        "kpis": ["Carbon Offset (tonnes)", "Energy Saved (kWh)", "Payback Period", "ESG Score"],
    },
}

investor_data = {
    "Angel Investor": {
        "focus": "early-stage potential, founding team strength, and unique idea",
        "ask_style": "seeking seed funding to validate the concept and build an MVP",
        "tone": "passionate and vision-driven",
        "typical_ask": "₹25L – ₹2Cr / $50K – $500K",
    },
    "Venture Capitalist (VC)": {
        "focus": "scalability, market size, traction, and 10x return potential",
        "ask_style": "seeking Series A funding to scale operations and expand market reach",
        "tone": "data-driven and growth-focused",
        "typical_ask": "₹2Cr – ₹50Cr / $1M – $10M",
    },
    "Government Grant": {
        "focus": "social impact, job creation, innovation, and national relevance",
        "ask_style": "applying for a government innovation grant to drive societal benefit",
        "tone": "impact-focused and policy-aligned",
        "typical_ask": "₹10L – ₹5Cr / $20K – $1M",
    },
    "Corporate / Strategic Investor": {
        "focus": "synergies, strategic alignment, IP value, and partnership potential",
        "ask_style": "seeking strategic investment and partnership to accelerate go-to-market",
        "tone": "partnership-oriented and ROI-focused",
        "typical_ask": "₹5Cr – ₹100Cr / $2M – $20M",
    },
    "Crowdfunding": {
        "focus": "community appeal, product story, and consumer validation",
        "ask_style": "launching a crowdfunding campaign to build community and raise initial capital",
        "tone": "emotional, story-driven, and community-focused",
        "typical_ask": "₹10L – ₹1Cr / $10K – $200K",
    },
}

# =============================================
#         PITCH DECK GENERATOR
# =============================================

def generate_pitch_deck(
    startup_name, tagline, founder_name, industry,
    investor_type, problem, solution, target_audience,
    funding_ask, use_of_funds, unique_value
):
    errors = []
    if not startup_name.strip(): errors.append("Startup Name")
    if not founder_name.strip(): errors.append("Founder Name")
    if not problem.strip(): errors.append("Problem Statement")
    if not solution.strip(): errors.append("Solution")
    if not target_audience.strip(): errors.append("Target Audience")
    if errors:
        return f"⚠️ Please fill in: {', '.join(errors)}", ""

    today = date.today().strftime("%B %d, %Y")
    ind = industry_data[industry]
    inv = investor_data[investor_type]
    tag = tagline.strip() if tagline.strip() else f"Revolutionizing {industry} with AI"
    ask = funding_ask.strip() if funding_ask.strip() else inv["typical_ask"]
    uof = use_of_funds.strip() if use_of_funds.strip() else "Product development, team hiring, and market expansion"
    uvp = unique_value.strip() if unique_value.strip() else f"A first-of-its-kind solution in the {industry} space"

    competitors_list = ", ".join(ind["competitors"])
    revenue_list = " | ".join(ind["revenue_models"])
    kpi_list = " | ".join(ind["kpis"])
    risk_list = " | ".join(ind["risks"])

    deck = f"""
╔══════════════════════════════════════════════════════════════════╗
         🚀  {startup_name.upper()} — INVESTOR PITCH DECK
         Generated on: {today}
╚══════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SLIDE 1 — COVER / TITLE SLIDE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🏢 Company Name   : {startup_name}
💡 Tagline        : "{tag}"
👤 Presented By   : {founder_name}
🏭 Industry       : {industry}
🎯 Investor Type  : {investor_type}
📅 Date           : {today}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SLIDE 2 — EXECUTIVE SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{startup_name} is a {industry} startup {inv["ask_style"]}.

We are building {solution} to solve the pressing problem of {problem}.
Our target customers are {target_audience}, and we offer {uvp}.

We are {inv["tone"]} in our approach, with a clear path to profitability
and a strong founding team committed to execution.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SLIDE 3 — PROBLEM STATEMENT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
❌ THE PROBLEM:
{problem}

🔍 WHY IT MATTERS:
- This problem directly affects {target_audience}
- Existing solutions are either too expensive, too complex, or outdated
- The gap in the market represents a massive untapped opportunity
- Current players ({competitors_list}) do not fully address this need

📊 PAIN POINTS:
- High cost of existing alternatives
- Poor user experience and lack of personalization
- No integrated solution currently available
- Growing demand with no adequate supply

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SLIDE 4 — OUR SOLUTION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ THE SOLUTION:
{solution}

🌟 UNIQUE VALUE PROPOSITION:
{uvp}

🔧 KEY FEATURES:
- AI-powered automation reducing manual effort by 70%
- Intuitive interface designed for {target_audience}
- Real-time analytics and intelligent recommendations
- Seamless integration with existing tools and workflows
- Scalable cloud-based architecture

💎 WHY WE ARE DIFFERENT:
- Purpose-built for the {industry} sector
- Faster, cheaper, and smarter than existing alternatives
- Built by domain experts who understand the problem firsthand

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SLIDE 5 — MARKET OPPORTUNITY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📈 MARKET SIZE:
- Total Addressable Market (TAM): {ind["market_size"]}
- Serviceable Addressable Market (SAM): ~15-20% of TAM
- Serviceable Obtainable Market (SOM): Targeting 1-2% in Year 1-2

🔥 KEY TRENDS DRIVING GROWTH:
{ind["trends"]}

🎯 TARGET SEGMENT:
- Primary: {target_audience}
- Secondary: Enterprises and institutions in the {industry} sector
- Geography: Starting domestic, expanding globally in Year 2

📊 MARKET TIMING:
- The market is at an inflection point
- Regulatory tailwinds favoring innovation
- Post-pandemic digital acceleration driving adoption

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SLIDE 6 — COMPETITOR ANALYSIS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🏆 COMPETITIVE LANDSCAPE:
Known Players: {competitors_list}

📊 COMPETITIVE COMPARISON:

  Feature               | Competitors  | {startup_name}
  ----------------------|--------------|----------------
  AI-Powered            | Partial      | ✅ Yes (Core)
  Affordable Pricing    | ❌ No        | ✅ Yes
  {industry} Specific  | ❌ No        | ✅ Yes
  Easy to Use           | ❌ Complex   | ✅ Intuitive
  Real-time Analytics   | Limited      | ✅ Advanced
  Customer Support      | Generic      | ✅ Dedicated

🛡️ OUR COMPETITIVE MOAT:
- Proprietary AI models trained on domain-specific data
- Strong network effects — value increases with each user
- First-mover advantage in our specific niche
- Deep relationships with key industry stakeholders

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SLIDE 7 — REVENUE MODEL & MONETIZATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💰 REVENUE STREAMS:
{revenue_list}

📊 PRICING STRATEGY:
- Starter Plan  : Free / Freemium (User acquisition)
- Growth Plan   : Mid-tier subscription (Core revenue)
- Enterprise    : Custom pricing with SLA (High value)
- Add-ons       : Premium features, API access, Analytics

📈 FINANCIAL PROJECTIONS:
  Year 1: Revenue focus — Customer acquisition & product-market fit
          Projected Users: 1,000 – 5,000
          Projected Revenue: ₹20L – ₹50L

  Year 2: Growth phase — Scaling and partnerships
          Projected Users: 10,000 – 50,000
          Projected Revenue: ₹1Cr – ₹5Cr

  Year 3: Expansion — New markets and enterprise deals
          Projected Users: 1,00,000+
          Projected Revenue: ₹10Cr – ₹25Cr

🔑 KEY METRICS TO TRACK:
{kpi_list}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SLIDE 8 — GO-TO-MARKET STRATEGY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🚀 LAUNCH STRATEGY:

  Phase 1 — VALIDATE (Month 1-3):
  • Beta launch with 100 pilot users from {target_audience}
  • Collect feedback and iterate rapidly
  • Build case studies and testimonials

  Phase 2 — GROW (Month 4-9):
  • Digital marketing: SEO, LinkedIn, targeted ads
  • Strategic partnerships with {industry} associations
  • Referral program and community building
  • PR and media coverage in {industry} publications

  Phase 3 — SCALE (Month 10-18):
  • Enterprise sales team and B2B outreach
  • Geographic expansion to Tier 2 cities / new regions
  • Product line extension and feature expansion
  • Explore international markets

📣 MARKETING CHANNELS:
- Content Marketing & Thought Leadership
- Social Media (LinkedIn, Twitter, YouTube)
- Industry Events & Conferences
- Word-of-mouth & Referral Programs
- Strategic Partnerships & Integrations

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SLIDE 9 — TRACTION & VALIDATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 CURRENT TRACTION:
- MVP developed and tested with early adopters
- Positive feedback from {target_audience} pilot users
- Letters of Intent (LOIs) from initial partners
- Growing waitlist of interested customers

🏆 KEY MILESTONES ACHIEVED:
  ✅ Problem validated through 50+ customer interviews
  ✅ MVP built and deployed
  ✅ First 100 beta users onboarded
  ✅ Strategic advisor with {industry} expertise onboarded
  ✅ Initial revenue / paid pilots underway

🌟 SOCIAL PROOF:
- "This solves exactly the problem we face daily."
  — Early Beta User, {target_audience}
- Recognized by [Industry Body] as a promising startup
- Featured in [Publication] as a top {industry} innovation

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SLIDE 10 — TEAM
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
👥 FOUNDING TEAM:

  👤 {founder_name} — CEO & Co-Founder
     • Domain expert with deep knowledge of {industry}
     • Vision-setter and product strategist
     • Strong network in target customer segment

  👤 Co-Founder / CTO (To be named)
     • Technical lead with AI/ML expertise
     • Experience building scalable products
     • Previously worked at top {industry} firms

  👤 Co-Founder / CMO (To be named)
     • Growth and marketing specialist
     • Experience in B2B and B2C sales
     • Strong industry connections

🎓 ADVISORS:
- Senior industry veteran from {industry} sector
- Experienced startup mentor / ex-VC
- Legal & compliance expert

💪 WHY THIS TEAM:
- Combined experience of 20+ years in {industry}
- Unique mix of technical, business, and domain expertise
- Deeply passionate about solving this specific problem
- Proven track record of building and scaling products

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SLIDE 11 — RISK ANALYSIS & MITIGATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚠️ KEY RISKS & MITIGATION STRATEGIES:

  Risk 1: {ind["risks"][0]}
  ✅ Mitigation: Dedicated compliance team, regular audits,
     legal advisors onboard from Day 1

  Risk 2: {ind["risks"][1]}
  ✅ Mitigation: Multi-layer security architecture,
     regular penetration testing, insurance coverage

  Risk 3: {ind["risks"][2] if len(ind["risks"]) > 2 else "Market adoption challenges"}
  ✅ Mitigation: Phased rollout, strong pilot program,
     customer success team to drive adoption

  Risk 4: Funding / Cash Flow Risk
  ✅ Mitigation: Conservative burn rate, milestone-based
     spending, multiple revenue streams from Year 1

  Risk 5: Competition from Large Players
  ✅ Mitigation: Niche focus, agility, customer intimacy,
     and strong IP moat

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 SLIDE 12 — THE ASK & CLOSING
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💰 FUNDING ASK:
  We are {inv["ask_style"]}.
  Amount Sought: {ask}

📋 USE OF FUNDS:
  {uof}

  Breakdown:
  • 40% — Product Development & AI/Tech Infrastructure
  • 25% — Team Hiring (Engineering, Sales, Customer Success)
  • 20% — Marketing & Customer Acquisition
  • 10% — Operations & Legal / Compliance
  • 5%  — Contingency Reserve

🎯 WHAT WE OFFER INVESTORS:
  • Equity stake in a high-growth {industry} startup
  • Access to a {ind["market_size"]} market
  • Strong founding team with domain expertise
  • Clear path to profitability by Year 2-3
  • Potential for 10x-50x return on investment

🌟 OUR VISION:
  To become the #1 AI-powered platform for {industry},
  serving millions of {target_audience} globally.

📞 NEXT STEPS:
  1. Schedule a 30-minute discovery call
  2. Share detailed financial model on request
  3. Arrange product demo and pilot discussion
  4. Term sheet and due diligence process

  📧 Contact : {founder_name} | {startup_name}
  🌐 Website : www.{startup_name.lower().replace(' ', '')}.com
  📅 Date    : {today}

╔══════════════════════════════════════════════════════════════════╗
   Thank you for your time and consideration!
   Let's build the future of {industry} together. 🚀
╚══════════════════════════════════════════════════════════════════╝
"""
    return deck.strip(), deck.strip()


# =============================================
#            GRADIO UI
# =============================================

with gr.Blocks(theme=gr.themes.Soft(), title="AI Pitch Deck Generator") as app:

    gr.Markdown("""
    # 🚀 AI Startup Pitch Deck Content Generator
    ### Generate a full 12-slide professional investor pitch deck instantly — no API needed!
    """)

    with gr.Tabs():

        # ---- TAB 1: INPUT ----
        with gr.Tab("📝 Startup Details"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### 🏢 Basic Information")
                    startup_name = gr.Textbox(label="🏢 Startup Name", placeholder="e.g. FinSmart AI")
                    tagline = gr.Textbox(label="💡 Tagline (optional)", placeholder="e.g. Smart Finance for Everyone")
                    founder_name = gr.Textbox(label="👤 Founder Name", placeholder="e.g. Rahul Kumar")
                    industry = gr.Dropdown(
                        choices=list(industry_data.keys()),
                        label="🏭 Industry",
                        value="Tech / Software"
                    )
                    investor_type = gr.Dropdown(
                        choices=list(investor_data.keys()),
                        label="🎯 Investor Type",
                        value="Venture Capitalist (VC)"
                    )

                with gr.Column():
                    gr.Markdown("### 💼 Business Details")
                    problem = gr.Textbox(
                        label="❌ Problem You Are Solving",
                        placeholder="e.g. Small businesses lack affordable financial planning tools",
                        lines=2
                    )
                    solution = gr.Textbox(
                        label="✅ Your Solution",
                        placeholder="e.g. An AI-powered financial dashboard for SMEs",
                        lines=2
                    )
                    target_audience = gr.Textbox(
                        label="🎯 Target Audience",
                        placeholder="e.g. Small business owners, freelancers, startup founders"
                    )
                    unique_value = gr.Textbox(
                        label="🌟 Unique Value Proposition",
                        placeholder="e.g. Only platform combining AI + real-time bank data for SMEs",
                        lines=2
                    )
                    funding_ask = gr.Textbox(
                        label="💰 Funding Ask (optional)",
                        placeholder="e.g. ₹1 Crore / $200K"
                    )
                    use_of_funds = gr.Textbox(
                        label="📋 Use of Funds (optional)",
                        placeholder="e.g. 40% tech, 30% hiring, 20% marketing, 10% ops",
                        lines=2
                    )

            generate_btn = gr.Button("⚡ Generate Full 12-Slide Pitch Deck", variant="primary", size="lg")

        # ---- TAB 2: OUTPUT ----
        with gr.Tab("📊 Generated Pitch Deck"):
            gr.Markdown("### 📬 Your Full Pitch Deck")

            deck_preview = gr.Textbox(
                label="📄 Pitch Deck Preview (Read Only)",
                lines=30,
                interactive=False
            )

            editable_deck = gr.Textbox(
                label="✏️ Edit Your Pitch Deck",
                lines=30,
                interactive=True,
                placeholder="Your generated pitch deck will appear here for editing..."
            )

            with gr.Row():
                copy_btn = gr.Button("📋 Copy to Clipboard", variant="secondary")
                download_btn = gr.Button("📥 Download as .txt", variant="secondary")
                clear_btn = gr.Button("🗑️ Clear All", variant="stop")

            download_output = gr.File(label="📥 Download Your Pitch Deck")
            status_msg = gr.Markdown("")

    # ---- Button Logic ----

    generate_btn.click(
        fn=generate_pitch_deck,
        inputs=[
            startup_name, tagline, founder_name, industry,
            investor_type, problem, solution, target_audience,
            funding_ask, use_of_funds, unique_value
        ],
        outputs=[deck_preview, editable_deck]
    )

    def save_deck(text):
        if not text.strip():
            return None
        path = "/tmp/pitch_deck.txt"
        with open(path, "w") as f:
            f.write(text)
        return path

    download_btn.click(
        fn=save_deck,
        inputs=[editable_deck],
        outputs=[download_output]
    )

    clear_btn.click(
        fn=lambda: ("", "", "", "", "", "", "", "", "", "", "", ""),
        outputs=[
            startup_name, tagline, founder_name, problem,
            solution, target_audience, unique_value,
            funding_ask, use_of_funds, deck_preview, editable_deck, status_msg
        ]
    )

    copy_btn.click(
        fn=None,
        js="""
        () => {
            const textareas = document.querySelectorAll('textarea');
            let content = '';
            textareas.forEach(ta => {
                if (ta.value.includes('PITCH DECK')) {
                    content = ta.value;
                }
            });
            if (content) {
                navigator.clipboard.writeText(content);
                alert('✅ Pitch Deck copied to clipboard!');
            } else {
                alert('⚠️ Please generate a pitch deck first!');
            }
        }
        """
    )

app.launch(share=True)

/tmp/ipykernel_2248/4159762451.py:419: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="AI Pitch Deck Generator") as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2196ee393a8f2e20db.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
